# 11 · Local LLM Inference: Latency, Caches, and Quantization
**Time:** 45 minutes · **Prerequisites:** 07b, optional group `llm`

Run with `uv run --group llm python scripts/check_notebooks.py --execute
--include-manual --notebook 11 --download-budget-gb 2`.
The budget covers new model files; cached snapshots require no weight download.

Use a small pinned Qwen model to compare 4-bit and 8-bit **weight** quantization,
then full-precision and 4-bit **KV** caches. Distinguish prompt processing
(prefill), time to first token, and generation throughput. These are independent
costs. See [MLX-LM generation](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/generate.py).

We quantize each run from the same original weights, never from already
quantized weights. No quality ranking is assumed.

In [ ]:
import gc
import time
from pathlib import Path
import numpy as np
import mlx.core as mx
import mlx.nn as nn
from mlx_lm import load, stream_generate
from mlx_lm.models.cache import make_prompt_cache
from mlx_lm.sample_utils import make_sampler
from experiment_utils import model_path, write_record

source = model_path("qwen_small")
greedy = make_sampler(temp=0)


## 1. Fix the request and measurement boundaries
Model loading is excluded from inference latency. Warm up once. Prefill
reuse is measured after building a prefix cache; report that build cost
separately so the first request is not mistaken for a cache hit.
Use the exact token prefix of the same serialized prompt. A cache belongs to
one model, token prefix, and quantization setting and is mutated by generation.

In [ ]:
def measure(model, tokenizer, tokens, cache=None):
    mx.reset_peak_memory()
    start = time.perf_counter()
    first = None
    output_tokens, text = [], ""
    for response in stream_generate(model, tokenizer, prompt=tokens, max_tokens=32,
                                    sampler=greedy, prompt_cache=cache):
        if first is None:
            first = time.perf_counter() - start
        output_tokens.append(response.token)
        text += response.text
    elapsed = time.perf_counter() - start
    if first is None:
        raise RuntimeError("Model returned no generation response")
    return {"ttft_seconds": first, "elapsed_seconds": elapsed,
            "prompt_tokens": response.prompt_tokens, "prefill_tokens_per_second": response.prompt_tps,
            "generation_tokens": response.generation_tokens,
            "generation_tokens_per_second": response.generation_tps,
            "peak_memory_gb": mx.get_peak_memory() / 1e9,
            "text": text, "output_token_ids": output_tokens}

records = []
for weight_bits in [4, 8]:
    model, tokenizer = load(source)
    nn.quantize(model, group_size=64, bits=weight_bits)
    mx.eval(model.parameters())
    prompt = tokenizer.apply_chat_template([{"role":"user", "content":
        ("MLX arrays use unified memory. " * 60) + "\nWhat do MLX arrays use? Reply briefly."}],
        tokenize=True, add_generation_prompt=True)
    _ = measure(model, tokenizer, prompt)  # warm-up, excluded
    prefix_length = len(prompt) - 24
    for kv_bits in [None, 4]:
        for repeat in range(2):
            def new_cache(current_model, bits):
                cache = make_prompt_cache(current_model)
                return cache if bits is None else [c.to_quantized(group_size=64, bits=bits) for c in cache]
            cold = measure(model, tokenizer, prompt, cache=new_cache(model, kv_bits))
            prefix_cache = new_cache(model, kv_bits)
            start = time.perf_counter()
            logits = model(mx.array(prompt[:prefix_length])[None], cache=prefix_cache)
            mx.eval(logits, [c.state for c in prefix_cache])
            prefix_seconds = time.perf_counter() - start
            warm = measure(model, tokenizer, prompt[prefix_length:], cache=prefix_cache)
            if kv_bits is None:
                assert cold["output_token_ids"] == warm["output_token_ids"], "Cached and uncached greedy outputs differ"
            for mode, row in [("cold", cold), ("prefix_reuse", warm)]:
                records.append({"weight_bits": weight_bits, "kv_bits": kv_bits, "repeat": repeat,
                                "mode": mode, "prefix_build_seconds": prefix_seconds if mode == "prefix_reuse" else 0,
                                **row})
    del model, prefix_cache, logits
    gc.collect()
    mx.clear_cache()
for row in records:
    print({k:v for k,v in row.items() if k not in ("text", "output_token_ids")})


## 2. Read quality as well as speed
Quantization can change outputs. Identical output on one prompt proves little;
inspect every reply and expand to held-out task examples before choosing a
setting. Peak memory is process allocation measured by MLX, not total system RAM.
Two repeats demonstrate reporting, not a reliable hardware benchmark.
No speculative-decoding speedup is implied by this lesson.

In [ ]:
for row in records:
    if row['repeat'] == 0:
        print(row['weight_bits'], row['kv_bits'], row['mode'], repr(row['text']))
print(write_record("11-inference", models=["qwen_small"], seed=0,
    settings={"max_tokens":32, "temperature":0, "group_size":64, "repeats":2,
              "prompt_token_ids":prompt, "prefix_length":prefix_length},
    metrics={"measurements":records}))


## Exercises
1. Vary prompt length while holding output length fixed. Which timing grows?
2. Compare total prefix-build + first-request time against cold inference, then
   compute amortized cost over 10 requests sharing a prefix.
3. Add a held-out extraction dataset and report exact accuracy for each setting.
4. Why must a cache be invalidated after loading a different LoRA adapter?

Do not reuse a cache containing a previous answer as though it contains only
the system prefix. Clone or rebuild the prefix state for independent requests.